In [1]:
# Written by Une Butaite & Jose Carlos A. R.
# Date: 25/Jun/2025
# Example for the 0.67" NIR PLM.

import numpy as np
import ctypes
from PLMController import PLMController
import matplotlib.pyplot as plt

In [ ]:
MAX_FRAMES = 120  # Max RGB frames stored in plmctrl's memory. Each frame holds 24 holograms -- limited by your RAM.

# Offset of the PLM virtual monitor. (0, 0) is the top-left corner of your main screen.
# ( x0 = 2560, y0 = 0 ) puts the PLM monitor to the right of a QHD main screen.
x0 = 2560
y0 = 0

relativePath = r'..\bin\plmctrl.dll'

# Model-based init: choose '.67NIRalpha' or '.67NIRgamma' to select dimensions and NIR LUT variant.
plm = PLMController('.67NIRgamma', relativePath, x0, y0, MAX_FRAMES=MAX_FRAMES)
#PLMController will internally call set_nir_variant('gamma')

# Pull dimensions back from the controller
N, M = plm.N, plm.M
print(f'PLM dimensions: N={N}, M={M}, is_nir={plm.is_nir}, nir_variant={plm.nir_variant_name}')

PLM dimensions: N=904, M=800, is_nir=True, nir_variant=gamma


In [3]:
plm.set_windowed(True)  # Debug only -- suggested while testing. Use False for real experiments.
plm.show_debug_panel(True)  # Show the debug panel in the UI. Highly recommend having this, it shows current frame, hologram, and PLM status.   

# Start the UI
plm.start_ui()

In [ ]:
plm.play()  # Start reading from the screen in the configured mode. plm.stop() to halt playback.

In [ ]:
# Configure the PLM for HDMI. Run the configuration cells once per boot, section by section,
# leaving a short pause between commands.
HDMI = 1
DisplayPort = 2

PlayOnce = 0
Continuous = 1

play_mode = Continuous
connection_type = HDMI

In [ ]:
# Set source to Parallel RGB (0) and port width to 24 bits (1)
plm.set_source(0, 1)

In [ ]:
# Set port swap for ports 0 and 1 to ABC -> ABC
plm.set_port_swap(0, 0)
plm.set_port_swap(1, 0)

In [ ]:
# Set Pixel Mode. 1 for HDMI (Single Pixel), 2 for DisplayPort (Dual Pixel)
plm.set_pixel_mode(connection_type)

In [ ]:
# Lock the PLM to the video stream -- wait ~3 s after this
plm.set_connection_type(connection_type)

In [ ]:
# Set video pattern mode (used for reading the video stream) -- wait ~3 s after this
plm.set_video_pattern_mode()

In [ ]:
# Update the bit lookup-table with the play mode and connection type
plm.update_lut(play_mode, connection_type)

In [ ]:
# Bitpack and insert a single frame.
phase = np.zeros((24, M, N), dtype=np.float32)
phase[:, :M//2, :N//2] = 0.0
phase[:, :M//2, N//2:] = 0.2
phase[:, M//2:, :N//2] = 0.3
phase[:, M//2:, N//2:] = 0.9

plt.imshow(phase[0, :, :])

frame = plm.bitpack_holograms_gpu(phase)
plm.insert_frames(frame, 0, format=1)

In [ ]:
# Bitpack and insert one frame at a time (phase ramps).
numHolograms = 24

for i in range(MAX_FRAMES):
    a = np.linspace(0, i * 2 + 1, N, dtype=np.float32)[None, :]
    b = np.linspace(0, 0,         M, dtype=np.float32)[:, None]
    ph = np.mod(a + b, 1)
    phase = np.tile(ph[np.newaxis, :, :], (numHolograms, 1, 1))
    phase = np.ascontiguousarray(phase)

    plm.bitpack_and_insert_gpu(phase, i)

In [ ]:
# Bitpack every frame first, then insert them all at once.
numHolograms = 24

frames = np.zeros((MAX_FRAMES, *plm.frame_shape), dtype=np.uint8)

for i in range(MAX_FRAMES):
    a = np.linspace(0, 0,         N, dtype=np.float32)[None, :]
    b = np.linspace(0, i * 2 + 1, M, dtype=np.float32)[:, None]
    ph = np.mod(a + b, 1)
    phase = np.tile(ph[np.newaxis, :, :], (numHolograms, 1, 1))
    phase_np = np.ascontiguousarray(phase)

    phase_ptr = phase_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    frame_ptr = frames[i].ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))

    plm.bitpack_holograms_gpu_ptr(phase_ptr, frame_ptr, numHolograms)

plm.insert_frames(frames, 0, format=1)

In [ ]:
# Multiple holograms with random wedge (blazed grating) phases.
x = np.linspace(-1,      1,      N, dtype=np.float32)
y = np.linspace(-M / N,  M / N,  M, dtype=np.float32)
xx, yy = np.meshgrid(x, y)
wedge = lambda alpha, beta: alpha * xx + beta * yy

numHolograms = 24
phase = np.zeros((numHolograms, M, N), dtype=np.float32)

for j in range(MAX_FRAMES):
    print(f'Python: Generating bitpacked hologram #{j + 1}')
    for i in range(numHolograms):
        alpha = 50.0 * (float(np.random.rand()) - 0.5)
        beta  = 50.0 * (float(np.random.rand()) - 0.5)
        phase[i] = np.mod(wedge(alpha, beta), 2 * np.pi) / (2 * np.pi)

    plm.bitpack_and_insert_gpu(phase, j)

In [ ]:
# Display the inserted frames as a looping sequence.
sequence = np.arange(MAX_FRAMES, dtype=np.uint64)
plm.set_frame_sequence(sequence)
plm.start_sequence(MAX_FRAMES)

In [ ]:
plm.cleanup()